# Middleware

Middleware provides a powerful mechanism for adding cross-cutting concerns to functions without modifying the function implementation itself. Like middleware in web frameworks, middleware wraps function calls with a four-phase pattern:

1. **Preprocess** - Inspect and modify inputs before calling next
2. **Call Next** - Delegate to the next middleware or function
3. **Postprocess** - Process, transform, or augment outputs
4. **Continue** - Return or yield the final result

## What You'll Learn

1. Understanding middleware concepts
2. Using built-in middleware (cache)
3. Applying middleware to functions using the SDK
4. Applying middleware to function groups

## Prerequisites

- Completed [04_functions_and_tools.ipynb](./04_functions_and_tools.ipynb)
- Basic understanding of NeMo Agent toolkit functions


In [1]:
import getpass
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Load environment variables from .env file
load_dotenv()

# Check for NVIDIA API key
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - examples may fail")


✅ NVIDIA_API_KEY loaded


## 1. Understanding Middleware

Middleware components are first-class components in NeMo Agent toolkit that:

- Wrap a function's execution with preprocessing and postprocessing logic
- Can be applied to individual functions or entire function groups
- Execute in order, forming an "onion" structure
- Can short-circuit execution (such as cache hits)

### Common Use Cases

| Middleware Type | Purpose |
|-----------------|----------|
| **Cache** | Memoize function outputs to avoid redundant calls |
| **Logging** | Log inputs and outputs for debugging |
| **Rate Limiting** | Control function call frequency |
| **Authentication** | Validate credentials before execution |
| **Retry** | Automatically retry failed function calls |


## 2. Built-in Cache Middleware

The cache middleware memoizes function outputs based on input similarity. This is particularly useful for:

- Expensive API calls that return the same result for identical inputs
- Evaluation runs where you want to avoid redundant LLM calls
- Development and testing scenarios


In [2]:
from nat.middleware.cache_middleware import CacheMiddleware

# Create a cache middleware with custom settings
cache_middleware = CacheMiddleware(
    enabled_mode="always",       # Cache is always active (vs "eval" for evaluation only)
    similarity_threshold=1.0,    # Exact matching (1.0) or fuzzy matching (< 1.0)
    name="my_cache",
)

print(f"✅ Cache middleware created: {cache_middleware.computed_name}")
print(f"   enabled_mode: {cache_middleware.enabled_mode}")
print(f"   similarity_threshold: {cache_middleware.similarity_threshold}")


✅ Cache middleware created: my_cache
   enabled_mode: always
   similarity_threshold: 1.0


### Cache Middleware Options

| Option | Values | Description |
|--------|--------|-------------|
| `enabled_mode` | `"always"`, `"eval"` | When caching is active |
| `similarity_threshold` | `0.0` - `1.0` | Input matching threshold (1.0 = exact) |


### Demonstrating the Cache in Action

Let's see the cache middleware working directly. We'll use the built `CacheMiddlewareImpl` to show how caching works at the function level:


In [ ]:
import time

from nat.middleware.cache_middleware import CacheMiddlewareImpl
from nat.middleware.function_middleware import FunctionMiddlewareContext

# Create the ACTUAL cache middleware implementation
# This is the same class used internally by NeMo Agent Toolkit
cache = CacheMiddlewareImpl(enabled_mode="always", similarity_threshold=1.0)

# Track how many times our "expensive" function is actually called
# Using a list to avoid global statement (mutable container pattern)
call_counter = [0]

async def expensive_function(value):
    """Simulates an expensive function (like an API call or LLM invocation)."""
    call_counter[0] += 1
    time.sleep(0.1)  # Simulate 100ms of work
    return f"Computed result for '{value}' (call #{call_counter[0]})"

# Create the context (middleware needs function metadata for logging)
context = FunctionMiddlewareContext(
    name="expensive_function",
    config=None,
    description="A demo expensive function",
    input_schema=None,
    single_output_schema=type(None),
    stream_output_schema=type(None),
)

print("=" * 60)
print("DEMONSTRATING CacheMiddlewareImpl FROM cache_middleware.py")
print("=" * 60)

# First call - cache MISS
print("\n📍 CALL 1: 'What is the weather?'")
start = time.time()
result1 = await cache.function_middleware_invoke(
    "What is the weather?",
    expensive_function,  # The middleware calls this on cache miss
    context
)
elapsed1 = time.time() - start
print(f"  Result: {result1}")
print(f"  Time: {elapsed1:.3f}s | Function executions: {call_counter[0]}")
print(f"  Cache entries: {len(cache._cache)}")

# Second call - SAME input (cache HIT)
print("\n📍 CALL 2: 'What is the weather?' (same input)")
start = time.time()
result2 = await cache.function_middleware_invoke(
    "What is the weather?",
    expensive_function,
    context
)
elapsed2 = time.time() - start
print(f"  Result: {result2}")
print(f"  Time: {elapsed2:.3f}s | Function executions: {call_counter[0]}")
print("  ⚡ CACHE HIT - No function call, instant response!")

# Third call - DIFFERENT input (cache MISS)
print("\n📍 CALL 3: 'What is the time?' (different input)")
start = time.time()
result3 = await cache.function_middleware_invoke(
    "What is the time?",
    expensive_function,
    context
)
elapsed3 = time.time() - start
print(f"  Result: {result3}")
print(f"  Time: {elapsed3:.3f}s | Function executions: {call_counter[0]}")
print(f"  Cache entries: {len(cache._cache)}")

# Fourth call - First input again (cache HIT)
print("\n📍 CALL 4: 'What is the weather?' (first input again)")
start = time.time()
result4 = await cache.function_middleware_invoke(
    "What is the weather?",
    expensive_function,
    context
)
elapsed4 = time.time() - start
print(f"  Result: {result4}")
print(f"  Time: {elapsed4:.3f}s | Function executions: {call_counter[0]}")
print("  ⚡ CACHE HIT - Still returning the original cached result!")

print("\n" + "=" * 60)
print("🎉 SUMMARY")
print("=" * 60)
print("Total middleware calls: 4")
print(f"Actual function executions: {call_counter[0]}")
print(f"Cache hits: {4 - call_counter[0]}")
print("\nCache contents (serialized keys):")
for key in cache._cache:
    print(f"  '{key}' -> '{cache._cache[key][:50]}...'")


## 3. Applying Middleware to Functions

Use the `mw` parameter to attach middleware to any function:


In [3]:
from nat.tool.datetime_tools import CurrentTimeTool

# Create a function with middleware attached
time_tool = CurrentTimeTool(
    name="cached_time_tool",
    mw=[cache_middleware],  # Attach middleware using 'mw' parameter
)

print(f"✅ Function created: {time_tool.computed_name}")
print(f"   Middleware attached: {time_tool.middleware}")


✅ Function created: cached_time_tool
   Middleware attached: ['my_cache']


### Multiple Middleware

You can attach multiple middleware components. They execute in order (first to last for preprocessing, last to first for postprocessing):


In [4]:
# Create another middleware (using the same cache config for demo)
eval_cache = CacheMiddleware(
    enabled_mode="eval",  # Only cache during evaluation
    name="eval_cache",
)

# Attach multiple middleware
tool_with_multiple_mw = CurrentTimeTool(
    name="multi_mw_tool",
    mw=[cache_middleware, eval_cache],  # Order matters!
)

print(f"✅ Function with multiple middleware: {tool_with_multiple_mw.middleware}")


✅ Function with multiple middleware: ['my_cache', 'eval_cache']


## 4. Complete Workflow Example

Let's create a complete workflow with middleware-enabled functions:


In [5]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# Create function with cache middleware
cached_tool = CurrentTimeTool(
    name="time_tool",
    mw=[cache_middleware],
)

# Create agent with the cached tool
agent = NatReActAgent(
    llm=llm,
    tools=[cached_tool],
    verbose=True,
)

# Create workflow
workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created with middleware-enabled function")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Workflow created with middleware-enabled function


### View Generated Configuration

Let's see how middleware appears in the generated YAML:


In [6]:
# Save and display the configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "middleware_example.yaml"
workflow.save_to_config_file(config_path)

print("Generated YAML configuration:\n")
with open(config_path) as f:
    print(f.read())


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Generated YAML configuration:

functions:
  time_tool:
    _type: current_datetime
    middleware:
    - my_cache

middleware:
  my_cache:
    _type: cache
    enabled_mode: always
    similarity_threshold: 1.0

llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - time_tool



### Run the Agent

Let's run the agent to demonstrate the workflow with middleware. 

> **Note**: The cache middleware caches based on the **function input**, not the agent prompt. Since `CurrentTimeTool` takes a parameter from the LLM, caching depends on whether the LLM passes identical inputs to the function.


In [7]:
# Run the agent - middleware is active and will cache function results
print("Running agent with middleware-enabled function:")
result = await workflow.prompt("What time is it right now?")
print(f"🤖 Response: {result}")


Running agent with middleware-enabled function:
🤖 Response: The current time is 2025-12-27 14:38:19 +0000.


### Understanding Cache Behavior

The cache middleware works at the **function level**, not the agent level:

- **Cache key**: Serialized function input (what the LLM passes to the tool)
- **Cache value**: The function's output

For tools that the LLM calls with consistent inputs, the cache will hit on subsequent calls. This is most useful for:
- **Evaluation runs**: Avoid redundant LLM/API calls during batch evaluation
- **Expensive operations**: Cache results from slow or costly function calls
- **Idempotent functions**: Functions where the same input always produces the same output


In [9]:
# Verify the middleware is attached to the function
print("Middleware configuration:")
print(f"  Function: {cached_tool.computed_name}")
print(f"  Middleware: {cached_tool.middleware}")
print(f"  Cache mode: {cache_middleware.enabled_mode}")
print(f"  Similarity threshold: {cache_middleware.similarity_threshold}")


Middleware configuration:
  Function: time_tool
  Middleware: ['my_cache']
  Cache mode: always
  Similarity threshold: 1.0


## 5. YAML Configuration Reference

Middleware can also be configured directly in YAML:

```yaml
# Define middleware components
middleware:
  my_cache:
    _type: cache
    enabled_mode: always
    similarity_threshold: 1.0

# Apply middleware to functions
functions:
  my_tool:
    _type: current_datetime
    middleware:
      - my_cache  # Reference by name

# Or apply to function groups
function_groups:
  my_group:
    _type: mcp_client
    middleware:
      - my_cache  # Applies to all functions in group
```


## CLI Commands

```bash
# Run workflow with middleware
nat run --config_file configs/middleware_example.yaml --input "What time is it?"
```

## Summary

✅ **Middleware concepts** - Preprocessing and postprocessing function calls  
✅ **Cache middleware** - `CacheMiddlewareConfig` for memoization  
✅ **SDK integration** - Use `mw=[...]` parameter on functions  
✅ **YAML configuration** - `middleware` section with function references  

## Next Steps

- **[09_configuration_guide.ipynb](./09_configuration_guide.ipynb)** - YAML configuration deep dive
- **[10_evaluation.ipynb](./10_evaluation.ipynb)** - Evaluate agent quality
- **[Advanced Middleware Documentation](../../../docs/source/build-workflows/advanced/middleware.md)** - Custom middleware development
